In [1]:
#Project Setup & Data Generation
import pandas as pd
import numpy as np
import gc
import uuid

# 1. Secure Data Generation (Simulating Pakistan E-commerce)
def generate_secure_data(n_rows=10000):
    np.random.seed(42)
    
    # Simulate cities and common Pincodes in Pakistan
    cities = ['Karachi', 'Lahore', 'Islamabad', 'Faisalabad', 'Multan', 'Peshawar']
    
    data = {
        'order_id': [str(uuid.uuid4())[:8] for _ in range(n_rows)],
        'customer_name': [f"User_{i}" for i in range(n_rows)], # Anonymized by default
        'phone_number': [f"92300{np.random.randint(1000000, 9999999)}" for _ in range(n_rows)],
        'city': np.random.choice(cities, n_rows),
        'order_value': np.random.uniform(500, 15000, n_rows),
        'payment_method': np.random.choice(['Cash on Delivery', 'JazzCash', 'EasyPaisa', 'Credit Card'], n_rows, p=[0.8, 0.1, 0.05, 0.05]),
        'past_rto_count': np.random.poisson(0.5, n_rows) # Most have 0, some have 1-2
    }
    
    df = pd.DataFrame(data)
    
    # Secure Anonymization Function
    df['phone_number'] = df['phone_number'].apply(lambda x: x[:5] + "****" + x[-2:])
    
    return df

df = generate_secure_data()
print(f"Dataset Initialized: {df.shape[0]} rows. Memory Usage: {df.memory_usage().sum() / 1024**2:.2f} MB")

Dataset Initialized: 10000 rows. Memory Usage: 0.50 MB


In [3]:
#Risk-Scoring Engine (Vectorized)
def calculate_risk_vectorized(df):
    """Calculates RTO Risk Score (0-100) using vectorized logic."""
    # Base Risk
    df['risk_score'] = 0
    
    # 1. Payment Method Risk (COD is higher risk in Pakistan)
    df.loc[df['payment_method'] == 'Cash on Delivery', 'risk_score'] += 30
    
    # 2. Historical Behavior (Heavy weight)
    df['risk_score'] += (df['past_rto_count'] * 25).clip(0, 50)
    
    # 3. High Value Order Risk
    df.loc[df['order_value'] > 10000, 'risk_score'] += 15
    
    # Ensure it doesn't exceed 100
    df['risk_score'] = df['risk_score'].clip(0, 100)
    return df

df = calculate_risk_vectorized(df)

In [4]:
#Agentic Simulation (Object-Oriented)
class LogisticsAgent:
    """Simulates an AI Agent handling order verification."""
    
    def __init__(self, agent_name="Shield-01"):
        self.agent_name = agent_name
        self.logs = []

    def process_order(self, row):
        score = row['risk_score']
        order_id = row['order_id']
        
        if score >= 80:
            action = "🔴 TRIGGER_VOICE_IVR"
            reason = "Critical Risk: High RTO History & COD"
        elif score >= 50:
            action = "🟡 SEND_WHATSAPP_OTP"
            reason = "Medium Risk: High Value COD"
        else:
            action = "🟢 AUTO_APPROVE"
            reason = "Low Risk: Digital Payment / Clean History"
            
        return action

# Applying the agent logic efficiently
shield_agent = LogisticsAgent()
df['agent_decision'] = df.apply(shield_agent.process_order, axis=1)

# Cleanup to save RAM for the next cell
gc.collect()

# Display results
df[['order_id', 'city', 'risk_score', 'agent_decision']].head(10)

,order_id,city,risk_score,agent_decision
0,7a137c11,Peshawar,45,🟢 AUTO_APPROVE
1,daec799d,Lahore,30,🟢 AUTO_APPROVE
2,e383ead8,Lahore,30,🟢 AUTO_APPROVE
3,c442c2d4,Faisalabad,45,🟢 AUTO_APPROVE
4,d4f1e5e9,Peshawar,30,🟢 AUTO_APPROVE
5,ad3a65da,Karachi,30,🟢 AUTO_APPROVE
6,bb44b625,Islamabad,95,🔴 TRIGGER_VOICE_IVR
7,ec7d5988,Faisalabad,70,🟡 SEND_WHATSAPP_OTP
8,2058dac6,Multan,30,🟢 AUTO_APPROVE
9,469d3563,Peshawar,55,🟡 SEND_WHATSAPP_OTP


In [5]:
# 1. Calculate the Business Impact
total_orders = len(df)
high_risk_count = len(df[df['risk_score'] >= 80])
savings_estimate = high_risk_count * 250 # Assume 250 PKR saved per prevented RTO

print("--- RTO-SHIELD PERFORMANCE REPORT ---")
print(f"✅ Total Orders Screened: {total_orders}")
print(f"🚨 High-Risk Orders Flagged: {high_risk_count}")
print(f"💰 Estimated Shipping Loss Prevented: {savings_estimate} PKR")
print("-------------------------------------")

# 2. Final Memory Flush for your Dell 5490
del df
gc.collect()
print("System Memory Optimized. Ready for next task.")

--- RTO-SHIELD PERFORMANCE REPORT ---
✅ Total Orders Screened: 10000
🚨 High-Risk Orders Flagged: 733
💰 Estimated Shipping Loss Prevented: 183250 PKR
-------------------------------------
System Memory Optimized. Ready for next task.


In [ ]:
#"I built RTO-Shield, a Python-based logistics engine designed for the Pakistani market. Using vectorized logic, I analyzed 10,000 simulated orders and successfully flagged 733 high-risk transactions. By triggering automated verification for only those specific orders, I estimated a potential saving of over 183,000 PKR in shipping losses while maintaining a seamless experience for low-risk customers."